# PolarisT atlas-profiled driver discovery demo

This notebook demonstrates the atlas-profiled PolarisT workflow using an example phenotype.

Complete [package installation](../README.md#1-package-installation) before running this notebook. Then follow the steps below.


## Data preparation

Download [Anndata_cd8_raw.h5ad](https://figshare.com/ndownloader/files/66953270) and save it in a local data directory. The dataset is documented in the associated [Figshare record](https://doi.org/10.6084/m9.figshare.32934569).

Set `resource_dir` below to your actual data directory. For example, if the file is saved as `/path/to/polarist_data/Anndata_cd8_raw.h5ad`, use `/path/to/polarist_data`.

**Note:** `resource_dir` must point to the directory containing the file, not to the `.h5ad` file itself.


In [ ]:
# Replace with the directory containing Anndata_cd8_raw.h5ad.
resource_dir = "/path/to/polarist_data"


In [1]:
from polarist import rank_seen_drivers

## Input parameters

- `phenotype_name`: phenotype label used in output filenames.
- `positive_genes`: genes expected to be highly expressed in the desired state.
- `negative_genes`: genes expected to be weakly expressed in the desired state; optional.
- `extreme_fraction`: fraction selected from each expression-score tail within every dataset. Valid range: `(0, 0.5]`. Default: `0.05`.
- `refinement_weight`: controls the strength of ranking refinement. Valid range: `[0, 1]`. Larger values preserve more of the original phenotype-alignment ranking, whereas smaller values apply stronger refinement. Default: `0.9`.
- `tf_only`: whether to additionally generate the transcription-factor subset in `result.tf_ranking`.
- `resource_dir`: local directory containing the downloaded `Anndata_cd8_raw.h5ad` file.


## Define the desired phenotype

This example defines a CD8+ T-cell objective with increased stemness and reduced exhaustion. Stemness-associated genes form the positive signature, while exhaustion-associated genes form the negative signature.

To define a custom phenotype, replace `positive_genes` and `negative_genes` and update `phenotype_name`. Use the same `resource_dir` set above. `negative_genes` can be omitted when using a positive signature alone.


In [2]:
phenotype_name = "stemness"
extreme_fraction = 0.05
refinement_weight = 0.9

positive_genes = [
    "TCF7", "LEF1", "SLAMF6", "SELL", "BCL2",
    "BCL6", "CXCR5", "CCNE1", "CCNE2",
]
negative_genes = ["TOX", "HAVCR2", "ENTPD1", "CD101", "CD244"]

## Rank atlas-profiled perturbations


In [3]:
result = rank_seen_drivers(
    positive_genes=positive_genes,
    negative_genes=negative_genes,
    phenotype_name=phenotype_name,
    extreme_fraction=extreme_fraction,
    refinement_weight=refinement_weight,
    tf_only=True,
    resource_dir=resource_dir,
)

[1/5] Scoring phenotype signatures and selecting the top 5.0% and bottom 5.0% cells within GEX...
[2/5] Phenotype cells selected; phenotype direction vector computed.
[3/5] Initial perturbation-phenotype alignment completed.
[4/5] Final perturbation ranking completed.
[5/5] Transcription-factor ranking completed (21 TFs).


## Full ranking

All atlas-profiled perturbations ranked toward the user-defined phenotype.

In [4]:
display(result.full_ranking.head(10))

,gene,dot_product,pearson_correlation,spearman_correlation,perturbation_direction,raw_rank,cluster_mean_score,refined_score,refined_rank
perturbation,,,,,,,,,
TCF7_Gain,TCF7,0.017247,0.794644,0.854950,gain,1.0,0.129754,0.728155,1.0
ARID1A_Loss,ARID1A,0.007381,0.707049,0.761513,loss,2.0,0.302099,0.666554,2.0
WT1_Gain,WT1,0.034400,0.703054,0.661846,gain,3.0,0.302099,0.662959,3.0
PIK3AP1_Gain,PIK3AP1,0.015714,0.660438,0.554616,gain,5.0,0.302099,0.624604,4.0
TNFRSF9_Gain,TNFRSF9,0.015176,0.651522,0.622247,gain,6.0,0.302099,0.616580,5.0
IL2RB_Gain,IL2RB,0.010831,0.667911,0.636485,gain,4.0,0.129754,0.614095,6.0
AKAP12_Gain,AKAP12,0.022559,0.645636,0.660957,gain,7.0,0.302099,0.611283,7.0
BTLA_Loss,BTLA,0.005849,0.639455,0.643604,loss,8.0,0.302099,0.605719,8.0
IL7R_Gain,IL7R,0.017908,0.629072,0.580423,gain,9.0,0.302099,0.596375,9.0


## Transcription-factor ranking

The subset of ranked perturbations whose target genes are included in `Human_tf_list.txt`.

In [5]:
display(result.tf_ranking.head(10))

,gene,dot_product,pearson_correlation,spearman_correlation,perturbation_direction,raw_rank,cluster_mean_score,refined_score,refined_rank
perturbation,,,,,,,,,
TCF7_Gain,TCF7,0.017247,0.794644,0.854950,gain,1.0,0.129754,0.728155,1.0
WT1_Gain,WT1,0.034400,0.703054,0.661846,gain,3.0,0.302099,0.662959,3.0
PRDM13_Gain,PRDM13,0.020354,0.619379,0.545717,gain,11.0,0.302099,0.587651,11.0
NFYB_Gain,NFYB,0.017594,0.478896,0.572414,gain,28.0,0.302099,0.461216,28.0
ATF6B_Gain,ATF6B,0.007637,0.433684,0.474972,gain,32.0,0.129754,0.403291,33.0
BATF_Gain,BATF,0.006767,0.347526,0.398888,gain,40.0,0.129754,0.325748,41.0
FOSB_Gain,FOSB,0.021864,0.323826,0.359288,gain,43.0,0.129754,0.304419,43.0
RELA_Gain,RELA,0.016441,0.278901,0.295217,gain,46.0,0.129754,0.263986,47.0
STAT6_Loss,STAT6,0.001324,0.252855,0.129700,loss,52.0,0.302099,0.257779,49.0


## Save the rankings

In [6]:
result.full_ranking.to_csv(
    f"atlas-profiled_{result.phenotype_name}_full_ranking.csv",
    index_label="Perturbation",
)
result.tf_ranking.to_csv(
    f"atlas-profiled_{result.phenotype_name}_tf_ranking.csv",
    index_label="Perturbation",
)